# 🔎 Lesson 11: Basic Retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/11_retrieval_basics.ipynb)

> **Retrieval is the **R** in RAG. If it brings the wrong chunks, even the best model can't give the right answer.**

---

## 🎯 Goal
By the end of this lesson you can write a `retrieve` function, choose `top_k`, combine it with metadata filters, and recognize when retrieval found nothing useful.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~25 minutes |
| 📚 **Prerequisites** | [Lesson 09](09_vector_databases.ipynb) |
| 🔑 **API key** | Not needed |
| 🛠 **Libraries** | `chromadb`, `sentence-transformers`, `numpy` |

In [ ]:
!pip install -q chromadb sentence-transformers numpy

## 🔄 Where we are

In Lesson 09 we stored vectors and searched them. **Retrieval** is that search, wrapped for RAG: *given a question, return the few chunks the model should read.*

Three decisions shape it:

| Decision | Question |
|---|---|
| **`top_k`** | how many chunks to return? |
| **Filter** | which chunks are allowed? (metadata) |
| **Cut-off** | what if even the best chunk isn't relevant? |

To make these visible, this lesson uses **smaller chunks** than before: 11 short ones, so an answer can be spread over several of them.

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb

EMBED_MODEL = "intfloat/multilingual-e5-small"
embedder = SentenceTransformer(EMBED_MODEL)

chunks = [
    {"id": "pto-0", "section": "PTO", "text": "Full-time employees receive paid time off based on years of service."},
    {"id": "pto-1", "section": "PTO", "text": "With less than 1 year of service, employees receive 10 PTO days per year."},
    {"id": "pto-2", "section": "PTO", "text": "With 1 to 3 years of service, employees receive 15 PTO days per year."},
    {"id": "pto-3", "section": "PTO", "text": "With more than 3 years of service, employees receive 20 PTO days per year."},
    {"id": "pto-4", "section": "PTO", "text": "PTO requests are submitted in advance through the HR system, unless related to an emergency."},
    {"id": "conduct-0", "section": "Conduct", "text": "Employees must maintain a respectful, professional environment free from harassment, discrimination and intimidation."},
    {"id": "conduct-1", "section": "Conduct", "text": "Disruptive behavior and verbal abuse are prohibited and may lead to disciplinary action."},
    {"id": "travel-0", "section": "Travel", "text": "Reasonable and necessary expenses of approved business trips are reimbursed."},
    {"id": "travel-1", "section": "Travel", "text": "Flights, hotels and meals must be recorded with receipts in the finance system."},
    {"id": "nda-0", "section": "NDA", "text": "Employees must protect confidential information such as product roadmaps, customer data and proprietary algorithms."},
    {"id": "nda-1", "section": "NDA", "text": "Confidential information must not be shared with unauthorized people, during or after employment."},
]

chunk_vectors = embedder.encode(["passage: " + chunk["text"] for chunk in chunks], normalize_embeddings=True)

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection(
    name="handbook_small",
    configuration={"hnsw": {"space": "cosine"}},
)
collection.upsert(
    ids=[chunk["id"] for chunk in chunks],
    embeddings=chunk_vectors.tolist(),
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[{"source": "pdf1.pdf", "section": chunk["section"], "position": position} for position, chunk in enumerate(chunks)],
)
print("stored:", collection.count())

d:\AI Source\Rag_torurial\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1986.77it/s]


stored: 11


## 🧩 Step 1: A `retrieve` function

We wrap the search in one function that returns **simple dictionaries**: the id, the section, the similarity and the text. Later steps (building the prompt, evaluating) will only need this shape.

Then we try a question whose answer is **in two different chunks**: the number of days (`pto-3`) and how to request them (`pto-4`).

In [2]:
def retrieve(question, top_k=3, where=None):
    question_vector = embedder.encode(["query: " + question], normalize_embeddings=True)[0]
    result = collection.query(
        query_embeddings=[question_vector.tolist()],
        n_results=top_k,
        where=where,
    )
    return [
        {"id": chunk_id, "section": metadata["section"], "similarity": 1 - distance, "text": text}
        for chunk_id, metadata, distance, text in zip(
            result["ids"][0], result["metadatas"][0], result["distances"][0], result["documents"][0]
        )
    ]

question = "How many PTO days do I get after five years, and how do I request them?"

for top_k in [1, 3]:
    print(f"--- top_k={top_k} ---")
    for item in retrieve(question, top_k=top_k):
        print(f'{item["similarity"]:.3f} | {item["id"]:10} | {item["text"][:70]}')

--- top_k=1 ---
0.896 | pto-2      | With 1 to 3 years of service, employees receive 15 PTO days per year.
--- top_k=3 ---
0.896 | pto-2      | With 1 to 3 years of service, employees receive 15 PTO days per year.
0.881 | pto-3      | With more than 3 years of service, employees receive 20 PTO days per y
0.872 | pto-1      | With less than 1 year of service, employees receive 10 PTO days per ye


### 👀 What we saw

Question: *"How many PTO days do I get after **five** years, and how do I request them?"*

| `top_k` | What came back | Problem |
|---|---|---|
| 1 | `pto-2` (**1 to 3 years**, 15 days) | wrong tier: five years is "more than 3" |
| 3 | `pto-2`, `pto-3`, `pto-1` | contains the right tier (`pto-3`), but **`pto-4`** (how to request) is missing, and `pto-1` (< 1 year) took its place |

The scores were `0.896`, `0.881`, `0.872`: **almost the same**. The model can hardly tell the right chunk from a similar wrong one.

Let's look at the whole ranking to see where the chunk we needed (`pto-4`) landed.

In [3]:
for rank, item in enumerate(retrieve(question, top_k=11), start=1):
    mark = "<-- needed" if item["id"] in ("pto-3", "pto-4") else ""
    print(f'{rank:2}. {item["similarity"]:.3f} | {item["id"]:10} {mark}')

 1. 0.896 | pto-2      
 2. 0.881 | pto-3      <-- needed
 3. 0.872 | pto-1      
 4. 0.842 | pto-4      <-- needed
 5. 0.821 | pto-0      
 6. 0.791 | nda-0      
 7. 0.788 | nda-1      
 8. 0.780 | conduct-0  
 9. 0.778 | travel-1   
10. 0.774 | travel-0   
11. 0.767 | conduct-1  


## 🧩 What can we do?

The question has **two parts**, and a single vector has to represent both. Some options (we will build them later in the course):

| Idea | How it helps | Lesson |
|---|---|---|
| Larger `top_k` | more chances to include the needed chunk, but more noise for the model | here |
| **Split the question** into sub-questions and retrieve for each | each sub-question points at one chunk | 20 |
| Better **chunking** (keep the three tiers in one chunk) | removes near-duplicates that crowd the list | 06 |
| **Reranking** | a second, more careful model re-orders the candidates | 19 |

Let's try the simplest one: retrieve for each part **separately**.

In [4]:
sub_questions = [
    "How many PTO days do I get after five years?",
    "How do I request PTO?",
]

for sub_question in sub_questions:
    print(sub_question)
    for item in retrieve(sub_question, top_k=2):
        print(f'   {item["similarity"]:.3f} | {item["id"]:10} | {item["text"][:60]}')

How many PTO days do I get after five years?
   0.892 | pto-2      | With 1 to 3 years of service, employees receive 15 PTO days 
   0.876 | pto-3      | With more than 3 years of service, employees receive 20 PTO 
How do I request PTO?
   0.866 | pto-4      | PTO requests are submitted in advance through the HR system,
   0.856 | pto-2      | With 1 to 3 years of service, employees receive 15 PTO days 


### 👀 What we saw

The ranking of all 11 chunks for the question *"How many PTO days do I get after five years, and how do I request them?"*:

| Rank | Chunk | Similarity | Needed? |
|---|---|---|---|
| 1 | `pto-2` (1 to 3 years) | 0.896 | |
| 2 | `pto-3` (more than 3 years) | 0.881 | **yes** |
| 3 | `pto-1` (less than 1 year) | 0.872 | |
| 4 | `pto-4` (how to request) | 0.842 | **yes** |
| 5 | `pto-0` | 0.821 | |
| 6 to 11 | NDA, Conduct, Travel | 0.767 to 0.791 | |

Two things are visible:
- **The topic is found easily.** All five PTO chunks come first, and everything else is clearly lower.
- **Inside the topic, small details are hard.** The model barely separates "1 to 3 years" from "more than 3 years".

Splitting the question helped with the *second* part (`pto-4` became first for "How do I request PTO?"), but **not** with the numbers: for "after five years", `pto-2` was still first.

### 💡 A practical measure: did the needed chunk make it into the top `top_k`?

A model can read "more than 3 years" correctly **if that chunk reaches it**. So for retrieval, being **first** matters less than being **inside the top `top_k`**. We can check this directly:

In [5]:
needed = {"pto-3", "pto-4"}
ranking = [item["id"] for item in retrieve(question, top_k=11)]

for top_k in range(1, 6):
    found = needed & set(ranking[:top_k])
    print(f"top_k={top_k}: found {sorted(found)}  ({len(found)} of {len(needed)})")

top_k=1: found []  (0 of 2)
top_k=2: found ['pto-3']  (1 of 2)
top_k=3: found ['pto-3']  (1 of 2)
top_k=4: found ['pto-3', 'pto-4']  (2 of 2)
top_k=5: found ['pto-3', 'pto-4']  (2 of 2)


## 🎯 Metadata filters, and questions with no answer

**Filters.** With `where` we restrict the search to chunks whose metadata match. This is how you search "only in the Travel section", "only documents from 2025", "only for this customer".

**Questions with no answer.** Vector search **always returns the nearest chunks**, even when nothing is relevant. Let's see what that looks like:

- a filter that points at the **wrong** section on purpose,
- a question that is **not covered** by the handbook,
- a question that has **nothing to do** with it.

In [7]:
def show(items):
    for item in items:
        print(f'   {item["similarity"]:.3f} | {item["id"]:10} | {item["text"][:60]}')

flight_question = "How do I get my flight reimbursed?"

print("section = Travel:")
show(retrieve(flight_question, top_k=2, where={"section": "Travel"}))

print("section = NDA (wrong on purpose):")
show(retrieve(flight_question, top_k=2, where={"section": "NDA"}))

for out_of_scope in ["What is the dental insurance policy?", "What is the capital of France?"]:
    print(out_of_scope)
    show(retrieve(out_of_scope, top_k=2))

section = Travel:
   0.838 | travel-1   | Flights, hotels and meals must be recorded with receipts in 
   0.836 | travel-0   | Reasonable and necessary expenses of approved business trips
section = NDA (wrong on purpose):
   0.771 | nda-0      | Employees must protect confidential information such as prod
   0.728 | nda-1      | Confidential information must not be shared with unauthorize
What is the dental insurance policy?
   0.761 | nda-0      | Employees must protect confidential information such as prod
   0.752 | pto-4      | PTO requests are submitted in advance through the HR system,
What is the capital of France?
   0.709 | travel-1   | Flights, hotels and meals must be recorded with receipts in 
   0.700 | pto-4      | PTO requests are submitted in advance through the HR system,


### 👀 What we saw

**How many chunks do we need?**

| `top_k` | Needed chunks found |
|---|---|
| 1 | 0 of 2 |
| 2 and 3 | 1 of 2 (`pto-3`) |
| 4 and 5 | 2 of 2 |

For this two-part question, `top_k=1` finds nothing useful, and we need at least `top_k=4`. A bigger `top_k` also brings more noise to the model, so it is a trade-off.

**Filters and questions with no answer:**

| Query | Best similarity | Comment |
|---|---|---|
| flight question, filter `Travel` | 0.838 | right section |
| flight question, filter `NDA` (wrong on purpose) | 0.771 | the filter forced irrelevant chunks |
| "What is the dental insurance policy?" | 0.761 | not covered, still returned something |
| "What is the capital of France?" | 0.709 | unrelated, still returned something |
| relevant PTO question | 0.896 | the best-matching chunks we saw were around 0.82 to 0.90 |

Search **never says "nothing found"**. It returns the nearest chunks, however far they are. In our few examples the irrelevant ones scored lower (0.71 to 0.77) than the relevant ones (0.82 to 0.90), so a **cut-off** on the similarity could help. But look closely: the "dental" question (0.761) is almost as high as the wrong-filter result (0.771).

In [12]:
def retrieve_with_cutoff(question, top_k=4, min_similarity=0.80, where=None):
    items = retrieve(question, top_k=top_k, where=where)
    return [item for item in items if item["similarity"] >= min_similarity]

for test_question in [
    question,
    "What is the dental insurance policy?",
    "What is the capital of France?",
]:
    kept = retrieve_with_cutoff(test_question)
    print(test_question[:50], "->", [item["id"] for item in kept])

How many PTO days do I get after five years, and h -> ['pto-2', 'pto-3', 'pto-1', 'pto-4']
What is the dental insurance policy? -> []
What is the capital of France? -> []


In [13]:
def format_context(items):
    return "\n".join(f'[{item["id"]}] {item["text"]}' for item in items)

print(format_context(retrieve(question, top_k=4)))

[pto-2] With 1 to 3 years of service, employees receive 15 PTO days per year.
[pto-3] With more than 3 years of service, employees receive 20 PTO days per year.
[pto-1] With less than 1 year of service, employees receive 10 PTO days per year.
[pto-4] PTO requests are submitted in advance through the HR system, unless related to an emergency.


## ⚠️ Why `0.80` is only an example

The value `min_similarity=0.80` is **not a recommendation**. We picked it by looking at about a dozen scores from **one model** and **one tiny handbook**.

- Another model gives a different score range (Lesson 08: we saw a wrong answer at 0.854 with MiniLM).
- Another language or another set of documents shifts the scores too.
- With a cut-off, you risk throwing away a *correct* chunk that scored lower than expected.

A threshold should come from **measuring on your own questions**: some with answers, some without. That is part of the Evaluation lessons. Until then, a safer habit is: keep the top few chunks, and let the **prompt** tell the model to say *"I don't know"* when the context doesn't contain the answer (next lesson).

## 📌 Summary

- **Retrieval** = embed the question, find the `top_k` nearest chunks, optionally with a metadata filter.
- `top_k` is a trade-off: too small can miss a needed chunk (we needed 4), too big adds noise.
- A question with several parts can have its needed chunks spread out. **Splitting it** helped with one part but not with number details, and chunk design matters (compare with Lesson 06).
- Being **first** matters less than being **inside the top `top_k`**: check whether the needed chunks are found.
- Search **always returns something**. Scores can hint at relevance, but any threshold must be **measured**, not guessed.
- Return retrieved chunks in a **simple, uniform shape** (`id`, `section`, `similarity`, `text`): the next steps build on it.

## ✍️ Exercise (10 minutes)

1. Write **five more questions**: three that the handbook answers, two that it doesn't.
2. For each, print the best similarity with `retrieve(question, top_k=1)`.
3. Look at your numbers. Is there a value that separates the answered from the unanswered questions? Would you trust it on new questions?

**Success:** you can explain in one sentence why a threshold from five examples is risky.

💡 Solution: `solutions/11_retrieval_basics.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Make the retrieval for *"How many PTO days do I get after five years?"* put `pto-3` **first**, without changing the question.

Ideas to try (measure each one!): rewrite the chunk text so the tier is clearer (for example, repeat the section name in every chunk), or merge the three tiers into one chunk. Which change works? Which one breaks something else?

## 📖 Next

We have the right chunks. Now we give them to a language model, with a careful prompt, and ask it to answer **only from them**, cite them, and admit when it doesn't know. That is the **G** in RAG.

**➡️ Next lesson:** [12 Generation](12_generation.ipynb)